In [8]:
from __future__ import annotations

import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
import xml.etree.ElementTree as ET

import requests

FEED_URL = "https://www.data.jma.go.jp/developer/xml/feed/extra.xml"
REQUEST_TIMEOUT = 20
USER_AGENT = "Mozilla/5.0 (JMA-Typhoon-Pipeline)"
BULLETIN_CODES = ("VPTW", "VPWW53", "VPWW54")

CONE_POINTS = {
    12: 45,
    24: 80,
    48: 150,
    72: 225,
    96: 325,
    120: 450,
}


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "maps").exists():
            return candidate
    return current


def parse_iso_utc(value: str) -> datetime:
    return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(timezone.utc)


def parse_jma_coordinate(coord_text: str) -> tuple[float, float]:
    match = re.search(r"([+-]\d+(?:\.\d+)?)([+-]\d+(?:\.\d+)?)/?", coord_text.strip())
    if not match:
        raise ValueError(f"Unsupported coordinate format: {coord_text}")
    return float(match.group(1)), float(match.group(2))


def knots_to_kmh(knots: float | None) -> float | None:
    return round(knots * 1.852, 1) if knots is not None else None


def hpa_to_mmhg(hpa: float | None) -> float | None:
    return round(hpa * 0.75, 1) if hpa is not None else None


def icon_tag_for(wind_kts: float | None, point_type: str) -> str:
    if wind_kts is None:
        return "LOW.png"
    if wind_kts < 34:
        return "TDgray.png" if point_type == "past" else "TD.png"
    if wind_kts < 64:
        return "TSgray.png" if point_type == "past" else "TS.png"
    return "TYgray.png" if point_type == "past" else "TY.png"


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return r * c


def bearing_deg(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dlon = math.radians(lon2 - lon1)
    y = math.sin(dlon) * math.cos(p2)
    x = math.cos(p1) * math.sin(p2) - math.sin(p1) * math.cos(p2) * math.cos(dlon)
    return (math.degrees(math.atan2(y, x)) + 360) % 360


def bearing_to_compass16(deg: float) -> str:
    dirs = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
    idx = int((deg + 11.25) // 22.5) % 16
    return dirs[idx]


def interpolate_cone_radius(lead_hours: float) -> float:
    if lead_hours <= 12:
        return float(CONE_POINTS[12])
    if lead_hours >= 120:
        return float(CONE_POINTS[120])
    keys = sorted(CONE_POINTS)
    for left, right in zip(keys, keys[1:]):
        if left <= lead_hours <= right:
            y1, y2 = CONE_POINTS[left], CONE_POINTS[right]
            ratio = (lead_hours - left) / (right - left)
            return round(y1 + ratio * (y2 - y1), 1)
    return float(CONE_POINTS[120])


In [9]:
def typhoon_entry_score(entry_id: str, link: str) -> int:
    blob = f"{entry_id} {link}"
    score = 0
    if "VPTW" in blob:
        score += 4
    if "VPWW54" in blob:
        score += 3
    if "VPWW53" in blob:
        score += 2
    if blob.lower().endswith(".xml"):
        score += 1
    return score


def get_typhoon_candidates(session: requests.Session) -> list[dict[str, str]]:
    res = session.get(FEED_URL, timeout=REQUEST_TIMEOUT)
    res.raise_for_status()

    root = ET.fromstring(res.content)
    ns = {"atom": "http://www.w3.org/2005/Atom"}

    entries = []
    for entry in root.findall("atom:entry", ns):
        entry_id = (entry.findtext("atom:id", default="", namespaces=ns) or "").strip()
        link_node = entry.find("atom:link", ns)
        link = (link_node.attrib.get("href", "") if link_node is not None else "").strip()
        updated = (entry.findtext("atom:updated", default="", namespaces=ns) or "").strip()
        title = (entry.findtext("atom:title", default="", namespaces=ns) or "").strip()

        blob = f"{entry_id} {link}"
        if not any(code in blob for code in BULLETIN_CODES):
            continue

        entries.append({"title": title, "id": entry_id, "link": link, "updated": updated, "score": typhoon_entry_score(entry_id, link)})

    entries.sort(key=lambda x: (x.get("score", 0), x.get("updated", "")), reverse=True)
    return entries


def parse_jma_typhoon_xml(xml_bytes: bytes) -> dict:
    root = ET.fromstring(xml_bytes)
    ns = {"j": "http://xml.kishou.go.jp/jmaxml1/"}

    body = root.find("j:Body", ns)
    if body is None:
        raise ValueError("Invalid JMA XML: missing Body")

    infos = body.findall("j:MeteorologicalInfo", ns)
    if not infos:
        raise ValueError("Invalid JMA XML: missing MeteorologicalInfo")

    storm_name = None
    international_number = None
    points = []

    for idx, info in enumerate(infos):
        advisory_time = info.findtext("j:DateTime", default="", namespaces=ns).strip()
        item = info.find("j:Item", ns)
        if item is None or not advisory_time:
            continue

        if storm_name is None:
            storm_name = item.findtext("j:TropicalCyclonePart/j:Name", default="", namespaces=ns).strip() or None
        if international_number is None:
            international_number = item.findtext("j:TropicalCyclonePart/j:InternationalNumber", default="", namespaces=ns).strip() or None

        coord_text = item.findtext("j:CenterPart/j:Coordinate", default="", namespaces=ns).strip()
        if not coord_text:
            continue

        lat, lon = parse_jma_coordinate(coord_text)
        wind_text = item.findtext("j:IntensityPart/j:MaximumWindSpeed", default="", namespaces=ns).strip()
        pressure_text = item.findtext("j:IntensityPart/j:CentralPressure", default="", namespaces=ns).strip()

        points.append({"advisory_time": advisory_time, "lat": lat, "lon": lon, "type": "current" if idx == 0 else "forecast", "wind_kts": float(wind_text) if wind_text else None, "pressure_hpa": float(pressure_text) if pressure_text else None, "source": "JMA"})

    if not points:
        raise ValueError("No parsable points found in JMA XML")

    return {"storm_id": international_number or "unknown", "storm_name": storm_name or "Unknown", "basin": "WP", "track": points}

In [10]:
def enrich_track(track: list[dict]) -> list[dict]:
    sorted_track = sorted(track, key=lambda x: parse_iso_utc(x["advisory_time"]))
    current_point = next((p for p in sorted_track if p.get("type") == "current"), None)
    current_dt = parse_iso_utc(current_point["advisory_time"]) if current_point else None

    prev = None
    for point in sorted_track:
        point["wind_kmh"] = knots_to_kmh(point.get("wind_kts"))
        point["pressure_mmhg"] = hpa_to_mmhg(point.get("pressure_hpa"))
        point["icon_tag"] = icon_tag_for(point.get("wind_kts"), point.get("type", "forecast"))

        if prev is None:
            point["forward_speed"] = 0.0
            point["direction"] = "N"
        else:
            prev_dt = parse_iso_utc(prev["advisory_time"])
            cur_dt = parse_iso_utc(point["advisory_time"])
            dt_hours = max((cur_dt - prev_dt).total_seconds() / 3600.0, 0.0)
            dist_km = haversine_km(prev["lat"], prev["lon"], point["lat"], point["lon"])
            point["forward_speed"] = round(dist_km / dt_hours, 1) if dt_hours > 0 else 0.0
            point["direction"] = bearing_to_compass16(bearing_deg(prev["lat"], prev["lon"], point["lat"], point["lon"]))

        if point.get("type") in {"past", "current"}:
            point["cone_radius_km"] = 0
        else:
            if current_dt is None:
                point["cone_radius_km"] = interpolate_cone_radius(12)
            else:
                lead_hours = (parse_iso_utc(point["advisory_time"]) - current_dt).total_seconds() / 3600.0
                point["cone_radius_km"] = interpolate_cone_radius(lead_hours)

        prev = point

    return sorted_track


def dedupe_track(track: list[dict]) -> list[dict]:
    merged = {}
    for p in track:
        key = (p.get("advisory_time"), round(float(p.get("lat", 0)), 4), round(float(p.get("lon", 0)), 4))
        merged[key] = p
    return list(merged.values())


def merge_with_existing(storm_doc: dict, db_file: Path) -> dict:
    new_track = storm_doc["track"]
    if db_file.exists():
        old_doc = json.loads(db_file.read_text(encoding="utf-8"))
        old_track = old_doc.get("track", [])
        for p in old_track:
            if p.get("type") == "current":
                p["type"] = "past"
        old_track = [p for p in old_track if p.get("type") != "forecast"]
        combined = old_track + new_track
        storm_doc["storm_name"] = storm_doc.get("storm_name") or old_doc.get("storm_name", "Unknown")
    else:
        combined = new_track

    combined = dedupe_track(combined)
    current_candidates = sorted([p for p in combined if p.get("type") == "current"], key=lambda x: parse_iso_utc(x["advisory_time"]))
    if current_candidates:
        final_current_time = current_candidates[-1]["advisory_time"]
        for p in combined:
            if p.get("type") == "current" and p.get("advisory_time") != final_current_time:
                p["type"] = "past"

    storm_doc["track"] = enrich_track(combined)
    storm_doc["updated_at"] = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    storm_doc["source"] = "JMA"
    return storm_doc


In [11]:
def select_first_parsable_storm(session: requests.Session, candidates: list[dict]) -> tuple[dict, dict, str]:
    errors = []
    for entry in candidates:
        xml_url = entry["link"] or entry["id"]
        if not xml_url:
            continue
        try:
            xml_res = session.get(xml_url, timeout=REQUEST_TIMEOUT)
            xml_res.raise_for_status()
            storm_doc = parse_jma_typhoon_xml(xml_res.content)
            return entry, storm_doc, xml_url
        except Exception as exc:
            errors.append(f"{xml_url}: {exc}")

    return {}, {"storm_id": "", "storm_name": "", "basin": "WP", "track": []}, ""


def run_pipeline() -> dict:
    project_root = find_project_root(Path.cwd())
    output_dir = project_root / "maps" / "tropical_cyclone" / "jma" / "database"
    output_dir.mkdir(parents=True, exist_ok=True)

    state_file = output_dir / "_pipeline_state.json"
    state = json.loads(state_file.read_text(encoding="utf-8")) if state_file.exists() else {}

    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    candidates = get_typhoon_candidates(session)
    if not candidates:
        return {"status": "empty", "reason": "no typhoon bulletin entries in feed", "track": []}

    entry, storm_doc, xml_url = select_first_parsable_storm(session, candidates)
    if not storm_doc.get("track"):
        return {"status": "empty", "reason": "no parsable typhoon track xml found", "track": []}

    entry_id = entry.get("id", "")
    if state.get("last_processed_entry_id") == entry_id:
        return {"status": "skipped", "reason": "entry already processed", "feed_entry_id": entry_id, "track": []}

    db_file = output_dir / f"{storm_doc['storm_id']}.json"
    merged_doc = merge_with_existing(storm_doc, db_file)

    temp_file = db_file.with_suffix(".json.tmp")
    temp_file.write_text(json.dumps(merged_doc, ensure_ascii=False, indent=2), encoding="utf-8")
    temp_file.replace(db_file)

    state["last_processed_entry_id"] = entry_id
    state_file.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "status": "ok",
        "feed_entry_id": entry_id,
        "xml_url": xml_url,
        "output_file": str(db_file),
        "track_points": len(merged_doc["track"]),
        "storm_id": merged_doc["storm_id"],
        "storm_name": merged_doc["storm_name"],
    }


result = run_pipeline()
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "status": "empty",
  "reason": "no parsable typhoon track xml found",
  "track": []
}
